In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

100%|██████████| 26.4M/26.4M [00:02<00:00, 11.0MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 175kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.22MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 10.5MB/s]


In [2]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

In [3]:
loss_fn = nn.CrossEntropyLoss()


In [4]:
optimizer = torch.optim.SGD(model.parameters(),lr=learning_rate)

In [5]:
def train_loop(dataloader,model,loss_fn,optimizer):
  size = len(dataloader.dataset)
  model.train()
  for batch, (X,y) in enumerate(dataloader):
    pred = model(X)
    loss = loss_fn(pred,y)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if batch%100 == 0:
      loss, current = loss.item(), batch*batch_size + len(X)
      print(f"Loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

def test_loop(dataloader,model,loss_fn):
  model.eval()
  size = len(dataloader.dataset)
  num_batches = len(dataloader)
  test_loss,correct = 0, 0

  with torch.no_grad():
    for X,y in dataloader:
      pred = model(X)
      test_loss += loss_fn(pred,y).item()
      correct+=(pred.argmax(1)==y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [6]:
epochs = 10
for t in range(epochs):
  print(f"Epoch {t+1}\n---------------------------------")
  train_loop(train_dataloader,model,loss_fn,optimizer)
  test_loop(test_dataloader,model,loss_fn)
print("Done!")

Epoch 1
---------------------------------
Loss: 2.299330 [   64/60000]
Loss: 2.285479 [ 6464/60000]
Loss: 2.272554 [12864/60000]
Loss: 2.268985 [19264/60000]
Loss: 2.258757 [25664/60000]
Loss: 2.235461 [32064/60000]
Loss: 2.236624 [38464/60000]
Loss: 2.207025 [44864/60000]
Loss: 2.195219 [51264/60000]
Loss: 2.180326 [57664/60000]
Test Error: 
 Accuracy: 58.9%, Avg loss: 2.169788 

Epoch 2
---------------------------------
Loss: 2.179874 [   64/60000]
Loss: 2.161087 [ 6464/60000]
Loss: 2.113911 [12864/60000]
Loss: 2.124064 [19264/60000]
Loss: 2.082571 [25664/60000]
Loss: 2.036239 [32064/60000]
Loss: 2.044500 [38464/60000]
Loss: 1.973678 [44864/60000]
Loss: 1.970865 [51264/60000]
Loss: 1.905724 [57664/60000]
Test Error: 
 Accuracy: 61.3%, Avg loss: 1.902622 

Epoch 3
---------------------------------
Loss: 1.941386 [   64/60000]
Loss: 1.896548 [ 6464/60000]
Loss: 1.788615 [12864/60000]
Loss: 1.823921 [19264/60000]
Loss: 1.725590 [25664/60000]
Loss: 1.679392 [32064/60000]
Loss: 1.690666 [